公共初始化 Cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
git clone https://github.com/wdongdong0708-jpg/BrainDecoding2.git /content/BrainDecoding || (cd /content/BrainDecoding && git pull)
cd /content/BrainDecoding
python3 scripts/colab_setup.py --venv-dir /content/proj-venv --local-cache-dir /content/local_cache
/content/proj-venv/bin/python -c "from transformers import AutoTokenizer, AutoModelForTextEncoding; AutoTokenizer.from_pretrained('t5-large'); AutoModelForTextEncoding.from_pretrained('t5-large')"

Colab #2：pallier2025_full（sub-01..sub-10 10 被试 90 runs，SimpleB2T run 1–6 / 7 / 8–9）

1) 一次性逐被试流式物化并打包单 .tar 存云盘（约 3.6 GB，峰值盘占 <10 GB）

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding/runs
PY=/content/proj-venv/bin/python

$PY -u -m braindecoding cache \
  --dataset pallier2025_full \
  --subjects sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10 \
  --split-policy simpleb2t \
  --pnpl-root /content/tmp_pnpl_raw/pallier2025_full \
  --include-test --cleanup-raw

$PY scripts/cache_archive.py pack \
  --modality pallier2025_full --cache-dir /content/local_cache \
  --archive-dir /content/drive/MyDrive/BrainDecoding/archives

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
ARCHIVE_DIR=/content/drive/MyDrive/BrainDecoding_v2/archives
PY=/content/proj-venv/bin/python

# 1) 逐被试流式物化并打包单 .tar（~3.6 GB）
$PY -u -m braindecoding cache --dataset pallier2025_full --subjects sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10 --split-policy simpleb2t --pnpl-root /content/tmp_pnpl_raw/pallier2025_full --include-test --cleanup-raw
$PY scripts/cache_archive.py pack --modality pallier2025_full --cache-dir /content/local_cache --archive-dir "$ARCHIVE_DIR"

In [ ]:
# 2) 新 GPU 实例解包
$PY scripts/cache_archive.py unpack --modality pallier2025_full --cache-dir /content/local_cache --archive-dir "$ARCHIVE_DIR" --skip-existing

2) 第 1 批 GPU 训练（seed=0：R1 → R3 → R4 + 留出 test 评估）

In [ ]:

# 3) 第 1 批：R1 -> R3 -> R4 + 留出 test 评估
$PY -u -m braindecoding train --config configs/training/pallier2025_full_word_encoder.yaml --exp-tag ch2_real_w10 --seed 0 --require-cache --resume auto
$PY -u -m braindecoding train --config configs/training/pallier2025_full_context_transformer.yaml --exp-tag ch2_real_frozen_w10 --word-exp-tag ch2_real_w10 --preextract-frozen-cnn --seed 0 --require-cache --resume auto
$PY -u -m braindecoding train --config configs/training/siamese_debiased/pallier2025_full_siamese_debiased_context.yaml --exp-tag ch2_real_debias_w10 --word-exp-tag ch2_real_w10 --seed 0 --require-cache --resume auto

$PY -u -m braindecoding eval-run --config configs/training/pallier2025_full_word_encoder.yaml --exp-tag ch2_real_w10 --seed 0 --split test --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/pallier2025_full_context_transformer.yaml --exp-tag ch2_real_frozen_w10 --word-exp-tag ch2_real_w10 --seed 0 --split test --control-config configs/controls/temporal_shift/temporal_shift_control.yaml --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/siamese_debiased/pallier2025_full_siamese_debiased_context.yaml --exp-tag ch2_real_debias_w10 --word-exp-tag ch2_real_w10 --seed 0 --split test --control-config configs/controls/temporal_shift/temporal_shift_control.yaml --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot